# InfantEmotionGen Evaluation

This notebook runs the final evaluation workflow for the SDXLPrimary and SD3.5 Medium model outputs. It uses the repository scripts under `evaluation/scripts/` and writes metrics to `evaluation/results/`.

Final reference dataset: `InfantEmotionGen/InfantEmotionGen_Dataset`

Metrics: FID, CLIP Agreement, CLIPScore, FER Accuracy, FER Macro F1

## 1. Setup

Run this notebook from the repository root. If running locally, activate the evaluation environment first:

```bash
conda activate infant-eval
```

If the Hugging Face datasets or models are private/gated, authenticate before running:

```bash
hf auth login
```

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_ROOT = Path.cwd()
print("Repository root:", REPO_ROOT)
assert (REPO_ROOT / "evaluation" / "scripts").is_dir(), "Run this notebook from the InfantEmotionGen repo root."

## 2. Install Evaluation Dependencies

Skip this cell if your environment is already set up.

In [ ]:
# Uncomment if dependencies are not installed.
# !python -m pip install -r evaluation/requirements.txt
# !python -m pip install -r evaluation/requirements_generation.txt
# !python -m pip install -r evaluation/requirements_eval.txt

## 3. Validate or Populate the External Reference Dataset

This creates/validates `evaluation/data/external_reference/` with 250 images each for `angry`, `crying`, and `happy`.

In [ ]:
def run(command, env=None):
    print("$", " ".join(command))
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    result = subprocess.run(command, cwd=REPO_ROOT, env=merged_env, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(command)}")

external_root = REPO_ROOT / "evaluation" / "data" / "external_reference"
if external_root.is_dir() and any(external_root.rglob("*.png")):
    run(["evaluation/scripts/populate_external_reference.sh", "--validate-only"])
else:
    run(["evaluation/scripts/populate_external_reference.sh"])
    run(["evaluation/scripts/populate_external_reference.sh", "--validate-only"])

## 4. Check Generated Image Counts

Final evaluation expects 100 images per class for each run.

In [ ]:
for run_name in ["sdxl_primary", "sd35_medium"]:
    print(f"\n{run_name}")
    for emotion in ["angry", "crying", "happy"]:
        folder = REPO_ROOT / "evaluation" / "generated" / run_name / emotion
        count = len(list(folder.glob("*.png"))) if folder.is_dir() else 0
        print(f"  {emotion}: {count} images")

## 5. Run Evaluation

If generated images are already complete, run only the metric scripts. This does not regenerate images.

In [ ]:
DEVICE = os.environ.get("DEVICE", "auto")
REAL_DIR = str(REPO_ROOT / "evaluation" / "data" / "external_reference")

eval_env = {
    "REAL_DIR": REAL_DIR,
    "DEVICE": DEVICE,
}

run(["evaluation/scripts/evaluate_generated_run.sh", "sdxl_primary"], env=eval_env)
run(["evaluation/scripts/evaluate_generated_run.sh", "sd35_medium"], env=eval_env)
run(["evaluation/scripts/compare_model_runs.sh"])

## Optional: Resume Generation Before Evaluation

Use this only if one of the model folders is missing generated images. The runner uses `--skip-existing`, so completed images are preserved.

In [ ]:
# Uncomment only if generation is incomplete.
# env = {
#     "REAL_DIR": str(REPO_ROOT / "evaluation" / "data" / "external_reference"),
#     "DEVICE": "mps",        # use "auto", "mps", "cuda", or "cpu"
#     "MIN_FREE_GB": "25",   # lower only if you intentionally accept the disk-space risk
# }
# run(["evaluation/scripts/run_full_model_evaluation.sh"], env=env)

## 6. Show Final Comparison

In [ ]:
comparison_md = REPO_ROOT / "evaluation" / "results" / "comparison.md"
print(comparison_md.read_text())

## 7. Copy JSON Results Into Tracked Logs

The `evaluation/results/` folder is ignored by git. This cell copies the JSON summaries into `logs/evaluation_results/` so the final metrics can be committed.

In [ ]:
import shutil

logs_root = REPO_ROOT / "logs" / "evaluation_results"
logs_root.mkdir(parents=True, exist_ok=True)

for run_name in ["sdxl_primary", "sd35_medium", "pipeline_test"]:
    source_dir = REPO_ROOT / "evaluation" / "results" / run_name
    if source_dir.is_dir():
        dest_dir = logs_root / run_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for json_path in source_dir.glob("*.json"):
            shutil.copy2(json_path, dest_dir / json_path.name)

parameter_counts = REPO_ROOT / "evaluation" / "results" / "parameter_counts.json"
if parameter_counts.is_file():
    shutil.copy2(parameter_counts, logs_root / "parameter_counts.json")

for path in sorted(logs_root.rglob("*.json")):
    print(path.relative_to(REPO_ROOT))